Import the required libraries 

In [76]:
import pandas as pd
import numpy as np
import io

In [77]:
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',500)

loading the data from source

In [78]:
df = pd.read_csv('employees.csv')

In [79]:
df.columns

Index(['First Name', 'Gender', 'Start Date', 'Last Login Time', 'Salary',
       'Bonus %', 'Senior Management', 'Team'],
      dtype='str')

understand the dataset

In [80]:
df.head(4)

,First Name,Gender,Start Date,Last Login Time,Salary,Bonus %,Senior Management,Team
0,Douglas,Male,8/6/1993,12:42 PM,97308,6.945,True,Marketing
1,Thomas,Male,3/31/1996,6:53 AM,61933,4.170,True,NaN
2,Maria,Female,4/23/1993,11:17 AM,130590,11.858,False,Finance
3,Jerry,Male,3/4/2005,1:00 PM,138705,9.340,True,Finance


check the missing values

In [81]:
df.isnull().sum()

First Name            67
Gender               145
Start Date             0
Last Login Time        0
Salary                 0
Bonus %                0
Senior Management     67
Team                  43
dtype: int64

drop the colmns where no data is available

In [82]:
df.dropna(axis='index',how='all',inplace=True)

Drop the unwanted rows to get a clean data 

In [83]:
df.dropna(subset=['First Name'],inplace=True)

In [84]:
df.dropna(subset=['Gender','Team','Senior Management'],inplace=True)

if salary is nan replace with average mean

In [85]:
df['Salary']=np.where(df['Salary'].isnull(),df['Salary'].mean(),df['Salary'])

drop the duplicates 

In [86]:
df.drop_duplicates(inplace=True)

convert the negative salary and bonus to positive 

In [87]:


# Fix Salary column
df['Salary'] = np.where(df['Salary'] < 0, df['Salary'].mean(), df['Salary'])

# Fix Bonus % column
df['Bonus %'] = np.where(df['Bonus %'] < 0, df['Bonus %'].mean(), df['Bonus %'])


In [88]:
df['Salary'].min()
df['Salary'].max()

np.float64(149908.0)

calculate the average salary for each group 

In [89]:
df['Avg_salary_Department']=df.groupby(['Team'])['Salary'].transform('mean').round(2)

discription of the whole data 

In [90]:
discription=df.describe()
discription

,Salary,Bonus %,Avg_salary_Department
count,764.000000,764.000000,764.000000
mean,90433.196335,10.148041,90433.196924
std,32864.665282,5.608733,2680.074815
min,35013.000000,1.015000,85849.100000
25%,62071.750000,5.193250,88066.400000
50%,90428.000000,9.658500,90520.400000
75%,118075.250000,14.965000,91724.820000
max,149908.000000,19.944000,94519.080000


In [143]:
discription.to_csv('description.csv')

In [118]:
df['Start Date']=pd.to_datetime(df['Start Date'])

Now get the breif information of the data

In [121]:
info=pd.DataFrame({
    'total_people':df['First Name'].count(),
    'total_male'  :df[df['Gender']=='Male'].shape[0],
    'total_Female'  :df[df['Gender']=='Female'].shape[0],
    'first_employee' : df['Start Date'].min(),
    'latest_employee' : df['Start Date'].max(),
    'average_salary':df['Salary'].mean().round(2),
    'average_bonus':df['Bonus %'].mean().round(3),
     'total_teams':df['Team'].value_counts().count()
      },index=[0]
)

write data to the csv file

In [122]:
info.to_csv('company_information.csv')

In [123]:
df['Team']=df['Team'].astype(str).str.strip()
dept=df.groupby(['Team'])

In [124]:
df['Team'].unique()

<StringArray>
[           'Marketing',              'Finance',      'Client Services',
                'Legal',              'Product',          'Engineering',
 'Business Development',      'Human Resources',                'Sales',
         'Distribution']
Length: 10, dtype: str

Now summary of each of the department 

In [140]:
dept_information = dept.agg(
    total_people=('First Name', 'count'),
    total_male=('Gender', lambda x: (x == 'Male').sum()),
    total_female=('Gender', lambda x: (x == 'Female').sum()),
    first_employee=('Start Date', 'min'),
    latest_employee=('Start Date', 'max'),
    average_salary=('Salary', lambda x: x.mean().round(2)),
    average_bonus=('Bonus %', lambda x: x.mean().round(2))
)


In [142]:
dept_information.to_csv('department_information')